# CSI-WiFormer — ablation trên Kaggle

1. **Add Input** → dataset SSHAR (thư mục chứa `room_*/esp/rx_00/...`), ví dụ `phanvanhai/csi-room-02-0707`.
2. **Settings** → Accelerator **GPU T4 x2** (hoặc P100), **Internet On** (để `git clone`).
3. Sửa ô *Cấu hình* → **Save Version → Save & Run All (Commit)** để chạy nền.
4. Kết quả trong tab **Output**: `results/<RUN_TAG>/...` và `results_<RUN_TAG>.zip`.

**Chia dữ liệu (SSHAR, theo SSHAR_ESP_protocol.md, 7:1:2 = 4480 / 640 / 1280):**
- `repeat`: theo lần lặp. dir00: rep 1–7 train, rep 8 val, rep 9–10 test. dir01/02: rep 5 test, rep 4 của một hướng (dir01 nếu pos lẻ, dir02 nếu pos chẵn) val, còn lại train.
- `user`: 7 người train / 1 người val / 2 người test (case 0: val 12, test 13, 14).
- `position`: test = 2/8 vị trí bị giữ lại; phần còn lại chia train:val = 7:1.
- Epoch tốt nhất chọn theo **val**, test chỉ đánh giá 1 lần ở cuối. **Dừng sớm** khi val acc không tăng sau 10 epoch (`--patience=10`, đặt `--patience=0` trong `EXTRA` để tắt). Optimizer mặc định: **Adam, lr = 1e-3** (WiFormer có warmup 5 epoch + cosine).

**Giới hạn 12 giờ/phiên:** 1 run WiFormer 100 epoch trên SSHAR-ESP mất ~2–3 giờ (T4), nên **1 phiên chỉ chạy được ~3–4 biến thể** (dừng sớm thường rút ngắn đáng kể, xem cột `epochs_trained` ở bảng tổng hợp).
Chia `VARIANTS` ra nhiều phiên; muốn gộp/tiếp tục thì thêm Output của phiên trước làm Input và đặt `PREV_RESULTS` (biến thể đã xong sẽ được bỏ qua).

## Cấu hình

In [ ]:
REPO_URL     = 'https://github.com/phanvanhai/wiformer.git'
BRANCH       = 'ablation'
PRIVATE_REPO = False      # True -> cần Kaggle Secret 'GITHUB_TOKEN'
CODE_DIR     = '/kaggle/working/cmr_har_code'   # xoá ở cuối để Output gọn
DATA_ROOT    = ''         # để trống -> tự tìm trong /kaggle/input (thư mục chứa room_*)
OUTPUT       = '/kaggle/working'

DATASET    = 'sshar_esp'  # sshar_esp | sshar_asus | xrf55 | ut_har
SPLIT_MODE = 'repeat'     # repeat | position | user
CASES      = [0]          # position: 0..3, user: 0..4; repeat: bỏ qua (chỉ chạy 1 lần)

# Biến thể ablation (python ablation.py --list):
#   core  : full wo_fft wo_conv wo_block1 wo_block2 wo_block3
#   design: k2_per_rx k2_pe_none k2_pe_rx_sub k3_learned k3_nopos k2_l1 k3_l2 no_norm all_ant win32 win128
#   scale : k1_flat4 k1_deep k2_l3 k3_l6 d96 d128 d128_k1deep   (tăng độ sâu/rộng)
VARIANTS = ['full', 'wo_block2', 'wo_block3']     # ~3 biến thể / phiên 12h
EXTRA    = '--epochs=100'                          # thêm tham số cho main.py, vd '--lr=5e-4'
# EXTRA  = '--epochs=2'                            # chạy thử nhanh

RUN_TAG      = 'ablation_v2'   # thư mục kết quả: results/<RUN_TAG>/<split>_c<case>/
PREV_RESULTS = ''              # vd '/kaggle/input/wiformer-ablation-v2/results/ablation_v2' để chạy tiếp

## Môi trường + dữ liệu

In [ ]:
!nvidia-smi
!nproc; free -g; df -h /kaggle/working | cat
!find /kaggle/input -maxdepth 4 -type d | head -40

In [ ]:
import os
from pathlib import Path

def find_sshar(root='/kaggle/input', max_depth=6):
    """Thư mục chứa các thư mục room_* (có <device>/rx_00 bên trong)."""
    for depth in range(1, max_depth + 1):
        for p in sorted(Path(root).glob('/'.join(['*'] * depth))):
            if p.is_dir() and p.name.startswith('room') and any(p.glob('*/rx_00')):
                return str(p.parent)
    return None

if not DATA_ROOT:
    DATA_ROOT = find_sshar() if DATASET.startswith('sshar') else '/kaggle/input'
    assert DATA_ROOT, 'Không tìm thấy room_*/<device>/rx_00 trong /kaggle/input — kiểm tra Add Input'
print('DATA_ROOT =', DATA_ROOT)
!ls {DATA_ROOT} | head -20

## Lấy code

In [ ]:
import subprocess
url = REPO_URL
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = REPO_URL.replace('https://', f'https://{token}@')
subprocess.run(['rm', '-rf', CODE_DIR], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, CODE_DIR], check=True)
!cd {CODE_DIR} && git log -1 --format='commit %h | %ci | %s' && ls
!pip install -q einops thop

## Chạy ablation

In [ ]:
import shutil, shlex, time

def run(cmd, cwd):
    """Chạy lệnh và in log ra notebook theo thời gian thực."""
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    return proc.wait()

RESULT_BASE = f'{OUTPUT}/results/{RUN_TAG}'
cases = CASES if SPLIT_MODE in ('position', 'user') else [0]
t0 = time.time()
for case in cases:
    tag = SPLIT_MODE if SPLIT_MODE == 'repeat' else f'{SPLIT_MODE}_c{case}'
    result_root = f'{RESULT_BASE}/{tag}'
    os.makedirs(result_root, exist_ok=True)

    # chạy tiếp từ Output phiên trước: copy các run đã xong, --skip_done sẽ bỏ qua
    prev = Path(PREV_RESULTS) / tag if PREV_RESULTS else None
    if prev and prev.is_dir():
        shutil.copytree(prev, result_root, dirs_exist_ok=True)
        print('copied previous results from', prev)

    print(f'\n===== {DATASET} | {tag} | {VARIANTS} =====', flush=True)
    run(['python', '-u', 'ablation.py',
         f'--data_root={DATA_ROOT}', f'--dataset={DATASET}',
         f'--result_root={result_root}', '--skip_done',
         '--variants', *VARIANTS,
         '--extra', f'--split_mode={SPLIT_MODE}', f'--case_split={case}', *shlex.split(EXTRA)],
        cwd=CODE_DIR)
print(f'\nTotal time: {(time.time() - t0) / 3600:.2f} h')

## Tổng hợp tất cả run đã có (kể cả từ phiên trước)

In [ ]:
import pandas as pd
for csv in sorted(Path(RESULT_BASE).glob('*/ablation_*.csv')):
    print('\n#', csv.relative_to(RESULT_BASE))
    df = pd.read_csv(csv)
    cols = ['variant', 'accuracy', 'delta_acc', 'f1_macro', 'best_epoch', 'epochs_trained', 'total_params', 'training_time']
    df = df[[c for c in cols if c in df]]
    for c in ('accuracy', 'delta_acc', 'f1_macro'):
        if c in df: df[c] = (100 * df[c]).round(2)
    print(df.to_string(index=False))

## Đóng gói Output

In [ ]:
zip_path = shutil.make_archive(f'{OUTPUT}/results_{RUN_TAG}', 'zip', f'{OUTPUT}/results', RUN_TAG)
print(zip_path, f'{os.path.getsize(zip_path) / 1e6:.1f} MB')
shutil.rmtree(CODE_DIR, ignore_errors=True)
!ls -R {OUTPUT}/results | head -60